# `signal_gate5.py` — Playground

Manual verification notebook for **Gate 5: Edge Check + EV** (rules only — zero Claude cost).

| Function | Status | Notes |
|---|---|---|
| `decide_gate5_signal(candidate, gate_results)` | ✅ built | Deterministic BUY/SKIP from momentum + Gate 3 + EV formula |

**Output:** `{passed, decision, win_probability, expected_value, edge, position_confidence, reason, trade_levels, gate_summary}`

Gate 5 runs **after** Gates 1–4 pass. It does not re-read news or call Claude. Win probability is mapped from structured upstream outputs; EV uses the fixed ~2:1 reward:risk from `build_trade_levels`. **BUY** only when `expected_value >= MIN_EDGE_PCT` (default 4% from `.env`).

| Setup | Typical result |
|---|---|
| score=3, BULLISH conf≥8, no caution | **BUY** — high EV |
| score=2, BULLISH conf=6, caution=True | **SKIP** — EV below threshold |
| Missing `price`/`atr` | `KeyError` — caller must supply scanner fields |

In [ ]:
import sys
import pathlib

gate5_dir = pathlib.Path('.').resolve()
if not (gate5_dir / 'signal_gate5.py').exists():
    gate5_dir = pathlib.Path('backend/02_intelligence/gate5_signal').resolve()

intelligence_dir = gate5_dir.parent
for p in [str(intelligence_dir), str(gate5_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from signal_gate5 import decide_gate5_signal

---
## Happy path — strong signals → BUY

Score 3 momentum, BULLISH conf 9, no caution. Expect `decision=BUY`, high EV, `position_confidence=HIGH`.

In [ ]:
strong_candidate = {
    'ticker': 'NVDA',
    'price': 875.50,
    'atr': 12.30,
    'score': 3,
}
strong_gates = {
    'gate1': {'passed': True},
    'gate2': {'passed': True},
    'gate3': {
        'passed': True,
        'direction': 'BULLISH',
        'confidence': 9,
        'caution': False,
        'key_reason': 'Strong earnings beat and raised guidance',
    },
    'gate4': {
        'passed': True,
        'action': 'PASS',
        'contradiction_type': 'none',
        'risk_level': 'NONE',
        'reason': 'NONE',
    },
}

decide_gate5_signal(strong_candidate, strong_gates)

---
## Variation — weak / marginal setup → SKIP

Score 2, minimum passing confidence, caution flag. Expect `decision=SKIP`, EV below 4%.

In [ ]:
weak_candidate = {
    'ticker': 'NVDA',
    'price': 875.50,
    'atr': 12.30,
    'score': 2,
}
weak_gates = {
    'gate3': {
        'passed': True,
        'direction': 'BULLISH',
        'confidence': 6,
        'caution': True,
        'key_reason': 'Mixed headlines from medium sources',
    },
}

decide_gate5_signal(weak_candidate, weak_gates)

---
## Side-by-side — compare strong vs weak on same ticker

In [ ]:
for label, cand, gates in [
    ('strong', strong_candidate, strong_gates),
    ('weak',   weak_candidate,   weak_gates),
]:
    r = decide_gate5_signal(cand, gates)
    print(f"{label:6} {r['decision']:<4} EV={r['expected_value']:.3f} "
          f"win_prob={r['win_probability']:.0%} conf={r['position_confidence']}")

---
## Failure path — missing price/atr

In [ ]:
try:
    decide_gate5_signal({'ticker': 'BAD'}, weak_gates)
except KeyError as e:
    print(f'Expected KeyError: {e}')

---
## Free-play